## The security hierarchy — how Unity Catalog governs access

Unity Catalog's permission model is a strict tree, and privileges granted at any level **inherit** to everything below.

```text
metastore
 └── catalog          ← GRANT here inherits to ALL schemas / tables / views
      └── schema       ← GRANT here inherits to ALL tables / views in the schema
           └── table / view / volume / function / model / MV
```

**Two privileges gate access at every level above an object:**

- **`USE CATALOG`** on a catalog — without it you can't even *see* the schemas inside.
- **`USE SCHEMA`** on a schema — without it you can't see the tables inside.

A `SELECT` on `silver.customers` alone is **useless** if the principal lacks `USE CATALOG` on `fintech_dev` and `USE SCHEMA` on `silver`. **This is the single most common exam trap** — any answer that omits `USE CATALOG` / `USE SCHEMA` is wrong.

**Inheritance shortcut:** `GRANT SELECT ON SCHEMA fintech_dev.silver` gives `SELECT` on every existing *and future* table in `silver`. Powerful, but a foot-gun — prefer narrower grants in production.

The bank has three groups, all starting with **zero** access: `analysts` (aggregated tables, no PII), `fraud_analysts` (masked PAN, all rows, fraud schema only), and `compliance` (full PII, but only their region). Every grant in this module adds exactly what a group needs — no more.
